In [3]:
# # Complete VEP Analysis Pipeline

# 1. ECOC decoding with LDA
# 2. Confusion matrix analysis
# 3. Contrast response modeling
# 4. MDS visualization

# ## 0. Imports

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import seaborn as sns

from scipy.optimize import curve_fit
from scipy.stats import norm

import joblib



In [4]:
# I extract participant from HELIOS_Participant_Sheet_Part_B_Session_1 with age data

AGE_DATA_SESSION1 = {
    '1005': 50.0,
    '1007': 31.0,
    '1014': 23.0,
    '1004': 33.0,
    '1020': 29.0,
    '1017': 46.0,
    '1016': 36.0,
    '1002': 43.0,
    '1023': 46.0,
    '1034': 60.0,
    '1008': 25.0,
    '1041': 49.0,
    '3011': 31.0,
    '1001': 25.0,
    '3005': 39.0,
    '1042': 39.0,
    '1039': 63.0,
    '2023': 49.0,
    '3007': 41.0,
    '3030': 54.0,
    '3016': 44.0,
    '3014': 34.0,
    '2020': 43.0,
    '1011': 37.0,
    '1024': 24.0,
    '3001': 54.0,
    '1038': 45.0,
    '2006': 25.0,
    '1021': 28.0,
    '2009': 49.0,
    '1010': 26.0,
    '3006': 33.0,
    '1046': 61.0,
    '2028': 29.0,
    '2026': 47.0,
    '2037': 45.0,
    '3027': 72.0,
    '1044': 62.0,
    '1028': 62.0,
    '3034': 48.0,
    '2017': 31.0,
    '1052': 53.0,
    '3039': 34.0,
    '3041': 35.0,
    '1026': 25.0,
    '1018': 25.0,
    '2029': 50.0,
    '2002': 62.0,
    '3008': 32.0,
}

In [5]:
# %% 1) Configuration

# Paths
DATA_DIR = Path("/Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ")
RESULTS_DIR = DATA_DIR / "ecoc_results_01"
RESULTS_DIR.mkdir(exist_ok=True)
        

# Analysis parameters
ANALYSIS_NAME = "luminance"      # "luminance", "L-M", or "S-cone"
ELECTRODE_SET = "64"       # "64", "PO", or "32"

SUBJECTS = [
    '1005','1007','1014','1004','1020','1017','1016','1002','1023','1034',
    '1008','1041','3011','1001','3005','1042','1039','2023','3007','3030',
    '3016','3014','2020','1011','1024','3001','1038','2006','1021','2009',
    '1010','3006','1046','2028','2026','2037','3027','1044','1028','3034',
    '2017','1052','3039','3041','1026','1018','2029','2002','3008'
]

# Time parameters
SAMPLE_RATE  = 256          # Hz
WINDOW_SIZE  = 5            # samples (≈20 ms at 256 Hz)
TIME_RANGE   = [0.05, 0.30] # seconds: 50–300 ms



# Contrast levels
S = {
    "luminance": [0.035, 0.07, 0.14, 0.28],
    "L-M"      : [0.008, 0.016, 0.032, 0.064],
    "S-cone"   : [0.06, 0.12, 0.24, 0.48]
}

CONTRASTS = np.array(S[ANALYSIS_NAME], float)
n_levels  = len(CONTRASTS)

# Colors for visualization
COLORS = {
    "luminance": ["#d9d9d9", "#999999", "#666666", "#262626"],
    "L-M"      : ["#d98c8c", "#b23333", "#800000", "#260000"],
    "S-cone"   : ["#9cbfff", "#6666cc", "#333399", "#000026"]
}



In [6]:
# file Loader

def load_subject_data(subject):
    fp = DATA_DIR / f"sub-{subject}_{ANALYSIS_NAME}_{ELECTRODE_SET}_data.npz"
    with np.load(fp, allow_pickle=False) as d:
        # (trials, channels, time)
        X = d["X"].astype(np.float32, copy=False)      
        y = d["y"].ravel().astype(np.int32, copy=False)
        times = d["times"].ravel().astype(np.float64, copy=False)
    
    if X.shape[-1] != times.size and X.shape[1] == times.size:
        X = np.transpose(X, (0, 2, 1))
        print(f"[{subject}] transposed X -> (trials, channels, time)")
    assert X.shape[-1] == times.size, "time mismatch"
    return X, y, times


In [7]:
# %% Helpers

# A) accuracy→distance (d′ or logit)
from scipy.stats import norm
def accuracy_to_distance(A, mode="dprime"):
    A = (A + A.T) / 2.0
    np.fill_diagonal(A, 0.5)
    eps = 1e-6
    if mode == "dprime":
        p = np.clip(A, 0.5+eps, 1-eps)
        dprime = np.sqrt(2) * norm.ppf(p)
        D = dprime - dprime.min()
    elif mode == "logit":
        acc = np.clip(A, 0.5+eps, 1-eps)
        S = np.log(acc/(1-acc))
        S = (S - S.min()) / (S.max() - S.min() + eps)
        D = S
    else:
        raise ValueError("mode must be 'dprime' or 'logit'")
    D = (D + D.T) / 2.0
    np.fill_diagonal(D, 0.0)
    return D

# B) classical 1-D MDS
def cmdscale_1d(D):
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n))/n
    B = -0.5 * J @ (D**2) @ J
    vals, vecs = np.linalg.eigh(B)
    x = vecs[:, -1] * np.sqrt(max(vals[-1], 0))
    return x

# C) align and 0–1 scale to match contrast ordering
def align_and_scale(x, contrasts, unit="0to1"):
    if np.corrcoef(x, contrasts)[0,1] < 0:
        x = -x
    x = x - x.min()
    if unit == "0to1":
        xmax = x.max()
        if xmax > 0: x = x / xmax
    elif unit == "z":
        x = (x - x.mean()) / (x.std() + 1e-12)
    return x

# D) CM → pairwise distance (via misclassification)
def cm_to_dissimilarity(cm, mode="dprime"):
    cm = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    S_sim = 0.5 * (cm + cm.T)
    A = 1.0 - S_sim
    np.fill_diagonal(A, 0.5)
    return accuracy_to_distance(A, mode=mode)

# E) Naka–Rushton (3-param)
def naka(c, Rmax, c50, n):
    c = np.asarray(c, float)
    return Rmax * (c**n) / (c**n + c50**n)

def fit_naka(contrasts, y, fix_n=None):
    c = np.asarray(contrasts, float); y = np.asarray(y, float)
    if fix_n is None:
        p0     = [1.0, np.median(c)/2.0, 2.0]
        bounds = ([0.0, c.min()/10.0, 1.0], [np.inf, c.max(), 6.0])
        popt, pcov = curve_fit(naka, c, y, p0=p0, bounds=bounds, maxfev=20000)
        return popt, pcov
    else:
        def naka_fixed(c_, Rmax, c50): return naka(c_, Rmax, c50, fix_n)
        p0     = [1.0, np.median(c)/2.0]
        bounds = ([0.0, c.min()/10.0], [np.inf, c.max()])
        popt2, pcov2 = curve_fit(naka_fixed, c, y, p0=p0, bounds=bounds, maxfev=20000)
        Rmax, c50 = popt2
        return (Rmax, c50, fix_n), pcov2


In [8]:
# %% ECOC: decode + save accuracy(t) and CM(t)
for subject in SUBJECTS:
    try:
        X, y, times = load_subject_data(subject)
        n_trials, n_channels, n_times = X.shape
        n_windows = n_times - WINDOW_SIZE + 1
        window_times = times[WINDOW_SIZE//2 : WINDOW_SIZE//2 + n_windows]

        accuracies = np.zeros(n_windows, float)
        cms = np.zeros((n_windows, n_levels, n_levels), float)

        clf = make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        )

        for w in range(n_windows):
            Xw = X[:, :, w:w+WINDOW_SIZE].mean(axis=2)
            cm_sum = np.zeros((n_levels, n_levels), float)
            fold_acc = []
            for rep in range(10):  # N_REPEATS
                cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=rep)
                for tr, te in cv.split(Xw, y):
                    clf.fit(Xw[tr], y[tr])
                    y_pred = clf.predict(Xw[te])
                    fold_acc.append((y_pred == y[te]).mean())
                    cm_sum += confusion_matrix(y[te], y_pred, labels=np.arange(n_levels))
            accuracies[w] = np.mean(fold_acc)
            row = cm_sum.sum(axis=1, keepdims=True); row[row==0]=1.0
            cms[w] = cm_sum / row

        out = RESULTS_DIR / f"sub-{subject}_ecoc_results.npz"
        np.savez(out, time_decoder=window_times, accuracies=accuracies, confusion_matrices=cms)
        print(f"[{subject}] saved {out.name}  (windows={n_windows})")
    except Exception as e:
        print(f"[{subject}] ECOC error: {e}")
        continue


[1005] saved sub-1005_ecoc_results.npz  (windows=252)
[1007] saved sub-1007_ecoc_results.npz  (windows=252)
[1014] saved sub-1014_ecoc_results.npz  (windows=252)
[1004] saved sub-1004_ecoc_results.npz  (windows=252)
[1020] saved sub-1020_ecoc_results.npz  (windows=252)
[1017] saved sub-1017_ecoc_results.npz  (windows=252)
[1016] saved sub-1016_ecoc_results.npz  (windows=252)
[1002] saved sub-1002_ecoc_results.npz  (windows=252)
[1023] saved sub-1023_ecoc_results.npz  (windows=252)
[1034] saved sub-1034_ecoc_results.npz  (windows=252)
[1008] saved sub-1008_ecoc_results.npz  (windows=252)
[1041] saved sub-1041_ecoc_results.npz  (windows=252)
[3011] saved sub-3011_ecoc_results.npz  (windows=252)
[1001] saved sub-1001_ecoc_results.npz  (windows=252)
[3005] saved sub-3005_ecoc_results.npz  (windows=252)
[1042] saved sub-1042_ecoc_results.npz  (windows=252)
[1039] saved sub-1039_ecoc_results.npz  (windows=252)
[2023] saved sub-2023_ecoc_results.npz  (windows=252)
[3007] saved sub-3007_ecoc_r

In [9]:
# %% Per-subject: choose best window in TIME_RANGE, MDS→NR fit (n fixed)
for subject in SUBJECTS:
    try:
        with np.load(RESULTS_DIR / f"sub-{subject}_ecoc_results.npz", allow_pickle=False) as ed:
            t = ed["time_decoder"].ravel()
            acc = ed["accuracies"].ravel()
            cms = ed["confusion_matrices"]

        inr = (t >= TIME_RANGE[0]) & (t <= TIME_RANGE[1])
        idx = np.argmax(acc[inr]) if np.any(inr) else np.argmax(acc)
        widx = np.where(inr)[0][idx] if np.any(inr) else idx

        D  = cm_to_dissimilarity(cms[widx], mode="dprime")
        x1 = align_and_scale(cmdscale_1d(D), CONTRASTS)

         # keep n fixed for stability
        popt, _ = fit_naka(CONTRASTS, x1, fix_n=2.0)   
        Rmax, c50, n = map(float, popt)

        # plot
        c_line = np.linspace(CONTRASTS.min(), CONTRASTS.max(), 300)
        plt.figure()
        plt.scatter(CONTRASTS, x1, label="MDS data")
        plt.plot(c_line, naka(c_line, Rmax, c50, n), label=f"fit (n={n:.2f}, c50={c50:.3f})")
        plt.title(f"sub-{subject}  @ {t[widx]:.3f}s")
        plt.xlabel("Contrast"); plt.ylabel("Aligned MDS")
        plt.legend(); plt.tight_layout()
        plt.savefig(RESULTS_DIR / f"sub-{subject}_naka_rushton_fit.png", dpi=150, bbox_inches="tight")
        plt.close()

    except Exception as e:
        print(f"[{subject}] NR-fit error: {e}")
        continue


In [10]:
# %% Group decoding accuracy (mean±SEM) with chance line
import numpy as np, matplotlib.pyplot as plt, glob

def group_accuracy(results_dir, subjects=None):
    times, accs = None, []
    for f in sorted(glob.glob(str(results_dir / "sub-*_*ecoc_results.npz"))):
        sid = f.split("sub-")[1].split("_")[0]
        if subjects is not None and sid not in set(subjects): 
            continue
        with np.load(f, allow_pickle=False) as d:
            t = d["time_decoder"].ravel()
            a = d["accuracies"].ravel()
        if times is None:
            times = t
        else:
            if not np.allclose(times, t):
                raise ValueError(f"time mismatch in {f}")
        accs.append(a)
    A = np.vstack(accs)  # subjects × time
    m = np.nanmean(A, axis=0)
    sem = np.nanstd(A, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(A), axis=0).clip(min=1))
    return times, m, sem

t, m, sem = group_accuracy(RESULTS_DIR, subjects=SUBJECTS)
chance = 1.0 / len(CONTRASTS)

plt.figure(figsize=(8,4))
plt.title("Group average decoding accuracy")
plt.axhline(chance, ls="--", color="k", alpha=0.5, label=f"chance={chance:.2f}")
plt.plot(t, m, color="C3", lw=2)
plt.fill_between(t, m-sem, m+sem, color="C3", alpha=0.25)
plt.xlabel("Time (s)"); plt.ylabel("Accuracy")
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"{ANALYSIS_NAME}_group_accuracy.png", dpi=150, bbox_inches="tight")
plt.close()


In [11]:
# %% Heatmap helper 

def _fmt_contrast_labels(contrasts):
    return [f"{c:g}" for c in np.asarray(contrasts, float)]

def show_heatmap(M, title, xticklabels=None, yticklabels=None,
                 cmap="viridis", vmin=None, vmax=None, annotate=True,
                 cbar=True, ax=None, fmt=".2f"):
    if ax is None:
        fig, ax = plt.subplots()
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper", aspect="equal")
    if cbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if xticklabels is not None:
        ax.set_xticks(np.arange(len(xticklabels)))
        ax.set_xticklabels(xticklabels, rotation=0)
    if yticklabels is not None:
        ax.set_yticks(np.arange(len(yticklabels)))
        ax.set_yticklabels(yticklabels)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    if annotate:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, format(M[i, j], fmt),
                        ha="center", va="center",
                        color="white" if (vmin is not None and vmax is not None and
                                          (M[i,j]-vmin)/(vmax-vmin) > 0.5) else "black")
    return ax


In [12]:
# %% Per-subject CM & RDM heatmaps
def plot_subject_cm_and_rdm(subject, when="best", target_time=None, contrasts=CONTRASTS):
    """
    when="best": choose subject's best decoding time within TIME_RANGE (fallback global best)
    when="time": use the window closest to target_time (in seconds)
    """
    f = RESULTS_DIR / f"sub-{subject}_ecoc_results.npz"
    if not f.exists():
        print(f"[{subject}] missing ecoc_results"); return
    with np.load(f, allow_pickle=False) as d:
        t   = d["time_decoder"].ravel()
        acc = d["accuracies"].ravel()
        cms = d["confusion_matrices"]

    # choose window
    if when == "time" and target_time is not None:
        widx = int(np.argmin(np.abs(t - float(target_time))))
    else:
        inr = (t >= TIME_RANGE[0]) & (t <= TIME_RANGE[1])
        widx = np.where(inr)[0][int(np.argmax(acc[inr]))] if np.any(inr) else int(np.argmax(acc))

    cm = cms[widx]                                       # rows already normalized (your save)
    Ssym = 0.5 * (cm + cm.T)                             # symmetric confusion (similarity)
    D = cm_to_dissimilarity(cm, mode="dprime")           # representational *distance* (d′-like)

    labs = _fmt_contrast_labels(contrasts)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    show_heatmap(cm,   f"Confusion (t={t[widx]:.3f}s)", xticklabels=labs, yticklabels=labs,
                 cmap="Blues", vmin=0.0, vmax=1.0, ax=axes[0])
    show_heatmap(Ssym, "Symmetric confusion", xticklabels=labs, yticklabels=labs,
                 cmap="PuBu", vmin=0.0, vmax=1.0, ax=axes[1])
    vmaxD = np.nanmax(D) if np.isfinite(D).all() else None
    show_heatmap(D,    "RDM (d′ distance)", xticklabels=labs, yticklabels=labs,
                 cmap="magma", vmin=0.0, vmax=vmaxD, ax=axes[2])

    for ax in axes: ax.grid(False)
    plt.tight_layout()
    out = RESULTS_DIR / f"sub-{subject}_cm_rdm_{when}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[{subject}] saved {out}")

# run for all subjects at their own best time
for sid in SUBJECTS:
    plot_subject_cm_and_rdm(sid, when="best", contrasts=CONTRASTS)


[1005] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1005_cm_rdm_best.png
[1007] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1007_cm_rdm_best.png
[1014] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1014_cm_rdm_best.png
[1004] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1004_cm_rdm_best.png
[1020] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1020_cm_rdm_best.png
[1017] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1017_cm_rdm_best.png
[1016] saved /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_

In [13]:
# %% Group CM/RDM — align by GROUP peak time
def group_peak_time(subjects=SUBJECTS):
    T = None; A = []
    for sid in subjects:
        f = RESULTS_DIR / f"sub-{sid}_ecoc_results.npz"
        if not f.exists(): continue
        with np.load(f, allow_pickle=False) as d:
            t = d["time_decoder"].ravel()
            a = d["accuracies"].ravel()
        if T is None: T = t
        else:
            if not np.allclose(T, t): raise ValueError("time mismatch across subjects")
        A.append(a)
    A = np.vstack(A)
    inr = (T >= TIME_RANGE[0]) & (T <= TIME_RANGE[1])
    idx = np.where(inr)[0][int(np.argmax(np.nanmean(A[:, inr], axis=0)))] if np.any(inr) else int(np.argmax(np.nanmean(A, 0)))
    return T[idx], idx, T

def group_cm_rdm(align="group", contrasts=CONTRASTS):
    if align not in ("group","subject"):
        raise ValueError("align must be 'group' or 'subject'")
    # pick reference time index(es)
    if align == "group":
        tstar, idx_star, T = group_peak_time(SUBJECTS)
        print(f"group peak time: {tstar:.3f}s")
    # accumulate
    CMs = []; Ds = []
    for sid in SUBJECTS:
        f = RESULTS_DIR / f"sub-{sid}_ecoc_results.npz"
        if not f.exists(): continue
        with np.load(f, allow_pickle=False) as d:
            t = d["time_decoder"].ravel()
            acc = d["accuracies"].ravel()
            cms = d["confusion_matrices"]
        if align == "group":
            cm = cms[idx_star]
        else:  # each subject's own best
            inr = (t >= TIME_RANGE[0]) & (t <= TIME_RANGE[1])
            widx = np.where(inr)[0][int(np.argmax(acc[inr]))] if np.any(inr) else int(np.argmax(acc))
            cm = cms[widx]
        CMs.append(cm)
        Ds.append(cm_to_dissimilarity(cm, mode="dprime"))
    CMs = np.stack(CMs)
    Ds  = np.stack(Ds)

    CM_mean = np.nanmean(CMs, axis=0)
    D_mean  = np.nanmean(Ds,  axis=0)

    labs = _fmt_contrast_labels(contrasts)
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 4))
    show_heatmap(CM_mean, f"Mean confusion ({align}-aligned)",
                 xticklabels=labs, yticklabels=labs, cmap="Blues", vmin=0.0, vmax=1.0, ax=axes[0])
    vmaxD = np.nanmax(D_mean)
    show_heatmap(D_mean, "Mean RDM (d′ distance)",
                 xticklabels=labs, yticklabels=labs, cmap="magma", vmin=0.0, vmax=vmaxD, ax=axes[1])
    for ax in axes: ax.grid(False)
    plt.tight_layout()
    out = RESULTS_DIR / f"{ANALYSIS_NAME}_group_cm_rdm_{align}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print("Saved:", out)

# produce both versions
group_cm_rdm(align="group",   contrasts=CONTRASTS)
group_cm_rdm(align="subject", contrasts=CONTRASTS)


group peak time: 0.148s
Saved: /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/luminance_group_cm_rdm_group.png
Saved: /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/luminance_group_cm_rdm_subject.png


In [14]:
# %% Group MDS QC (mean ± SEM) 

import numpy as np
import matplotlib.pyplot as plt

def _load_cms_and_times(subject):
    """Try new ecoc_results first, then legacy confusion_matrices file."""
    f_new = RESULTS_DIR / f"sub-{subject}_ecoc_results.npz"
    if f_new.exists():
        with np.load(f_new, allow_pickle=False) as d:
            if "confusion_matrices" in d.files and "time_decoder" in d.files:
                return d["confusion_matrices"], d["time_decoder"].ravel(), "ecoc_results"
    f_old = RESULTS_DIR / f"sub-{subject}_confusion_matrices.npz"
    if f_old.exists():
        with np.load(f_old, allow_pickle=False) as d:
            return d["confusion_matrices"], d["time_windows"].ravel(), "confusion_matrices"
    return None, None, None

def _mds_qc(cms, contrasts):
    """Return rho (corr with contrast) and distcorr (D vs |x_i-x_j|) per window."""
    nW, nL, _ = cms.shape
    rho = np.zeros(nW); distcorr = np.zeros(nW)
    tri = np.triu_indices(nL, 1)
    for i in range(nW):
        D  = cm_to_dissimilarity(cms[i], mode="dprime")
        x  = cmdscale_1d(D)
        dR = np.abs(x[:,None] - x[None,:])
        distcorr[i] = np.corrcoef(D[tri], dR[tri])[0, 1]
        rho[i] = np.corrcoef(align_and_scale(x, contrasts, unit="0to1"), contrasts)[0, 1]
    return rho, distcorr

# collect per-subject curves
R_list, D_list, used, missing = [], [], [], []
t_ref = None

for sid in SUBJECTS:
    cms, t, src = _load_cms_and_times(sid)
    if cms is None:
        missing.append(sid)
        continue
    if t_ref is None:
        t_ref = t
    elif not np.allclose(t_ref, t):
        raise ValueError(f"time vector mismatch for sub-{sid}")
    rho, distcorr = _mds_qc(cms, CONTRASTS)
    R_list.append(rho); D_list.append(distcorr); used.append(sid)

if not R_list:
    print("No subjects found with CMs in RESULTS_DIR. Check paths or re-run ECOC.")
else:
    R = np.vstack(R_list)  # subjects × time
    D = np.vstack(D_list)

    r_m = np.nanmean(R, axis=0); r_sem = np.nanstd(R, axis=0, ddof=1)/np.sqrt(R.shape[0])
    d_m = np.nanmean(D, axis=0); d_sem = np.nanstd(D, axis=0, ddof=1)/np.sqrt(D.shape[0])

    plt.figure(figsize=(8,4))
    plt.title("MDS QC (group mean ± SEM)")
    plt.plot(t_ref, d_m, label="distance corr", lw=2)
    plt.fill_between(t_ref, d_m-d_sem, d_m+d_sem, alpha=0.20)
    plt.plot(t_ref, r_m, label="corr(x, contrast)", lw=2)
    plt.fill_between(t_ref, r_m-r_sem, r_m+r_sem, alpha=0.20)
    plt.axhline(0.90, ls="--", alpha=0.3); plt.axhline(0.70, ls="--", alpha=0.3)
    plt.axvline(0.0, color="k", lw=1, alpha=0.3)
    plt.xlabel("Time (s)"); plt.legend(frameon=False); plt.tight_layout()
    out = RESULTS_DIR / f"{ANALYSIS_NAME}_group_mds_qc.png"
    plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
    print(f"Saved: {out}")
    print(f"Included subjects: {len(used)}  Missing: {len(missing)}")
    if missing:
        print("Missing IDs:", ", ".join(missing))


Saved: /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/luminance_group_mds_qc.png
Included subjects: 49  Missing: 0


In [15]:
# %% Generate per-window NR timecourses and save as sub-<ID>_nr_timecourses.npz

import numpy as np

# ---- gain helpers (standalone; safe to re-declare) ----
def naka_derivative(c, Rmax, c50, n):
    c = np.asarray(c, float)
    num = Rmax * n * (c**(n-1)) * (c50**n)
    den = (c**n + c50**n)**2
    return num / den

def gain_at_c50(Rmax, c50, n):
    return (Rmax * n) / (4.0 * c50)

# ---- MDS from CMs (per window) ----
def mds_timecourses_from_cms(cms, window_times, contrasts):
    cms = np.asarray(cms, float)
    window_times = np.asarray(window_times, float).ravel()
    nW, nL, _ = cms.shape

    X = np.zeros((nW, nL))
    rho = np.zeros(nW)
    distcorr = np.zeros(nW)
    prange = np.zeros(nW)

    tri = np.triu_indices(nL, 1)

    for i in range(nW):
        D = cm_to_dissimilarity(cms[i], mode="dprime")
        x = cmdscale_1d(D)
        prange[i] = x.max() - x.min()
        dR = np.abs(x[:, None] - x[None, :])
        distcorr[i] = np.corrcoef(D[tri], dR[tri])[0, 1]
        xa = align_and_scale(x, contrasts, unit="0to1")
        rho[i] = np.corrcoef(xa, contrasts)[0, 1]
        X[i, :] = xa

    return X, rho, distcorr, prange, window_times

# ---- Fit NR per window (fixed n) and compute gain ----
def fit_nr_timecourse(X, contrasts, qc_mask, fix_n=2.0):
    nW, nL = X.shape
    Rmax = np.full(nW, np.nan)
    c50  = np.full(nW, np.nan)
    nvec = np.full(nW, fix_n if fix_n is not None else np.nan)
    gain_c50 = np.full(nW, np.nan)
    gain_each = np.full((nW, nL), np.nan)

    for i in range(nW):
        if not qc_mask[i]:
            continue
        try:
            popt, _ = fit_naka(contrasts, X[i, :], fix_n=fix_n)
            Ri, c50i, ni = map(float, popt)
            Rmax[i], c50[i], nvec[i] = Ri, c50i, ni
            gain_c50[i] = gain_at_c50(Ri, c50i, ni)
            gain_each[i, :] = naka_derivative(contrasts, Ri, c50i, ni)
        except Exception:
            # leave NaNs if fit fails
            pass

    return Rmax, c50, nvec, gain_c50, gain_each

# ---- QC thresholds (match what you used elsewhere) ----
QC_R_TH   = 0.70   # corr(x, contrast)
QC_D_TH   = 0.90   # MDS distance faithfulness
QC_PR_FR  = 0.20   # dynamic-range fraction (of subject's max)

# ---- Run and save per subject ----
for subject in SUBJECTS:
    try:
        f_ecoc = RESULTS_DIR / f"sub-{subject}_ecoc_results.npz"
        if not f_ecoc.exists():
            print(f"[{subject}] missing ecoc_results -> skip")
            continue

        with np.load(f_ecoc, allow_pickle=False) as d:
            cms   = d["confusion_matrices"]           # shape (nW, nL, nL)
            wtime = d["time_decoder"].ravel()

        # MDS timecourses
        X, rho, distcorr, prange, wtime = mds_timecourses_from_cms(cms, wtime, CONTRASTS)

        # QC mask
        mask = (rho >= QC_R_TH) & (distcorr >= QC_D_TH) & (prange >= QC_PR_FR * float(np.nanmax(prange)))

        # Per-window NR fit (n fixed)
        Rmax, c50, nvec, g_c50, g_each = fit_nr_timecourse(X, CONTRASTS, mask, fix_n=2.0)

        # Save
        out = RESULTS_DIR / f"sub-{subject}_nr_timecourses.npz"
        np.savez(
            out,
            time=wtime,
            X=X, rho=rho, distcorr=distcorr, prange=prange,
            qc_mask=mask.astype(np.int8),
            Rmax=Rmax, c50=c50, n=nvec,
            gain_c50=g_c50, gain_each=g_each
        )

        # quick verify
        with np.load(out, allow_pickle=False) as chk:
            assert "gain_c50" in chk.files and "gain_each" in chk.files
        print(f"[{subject}] saved {out.name}  | good windows: {int(mask.sum())}/{len(mask)}")

    except Exception as e:
        print(f"[{subject}] NR timecourse error: {e}")
        continue


[1005] saved sub-1005_nr_timecourses.npz  | good windows: 96/252
[1007] saved sub-1007_nr_timecourses.npz  | good windows: 38/252
[1014] saved sub-1014_nr_timecourses.npz  | good windows: 73/252
[1004] saved sub-1004_nr_timecourses.npz  | good windows: 111/252
[1020] saved sub-1020_nr_timecourses.npz  | good windows: 91/252
[1017] saved sub-1017_nr_timecourses.npz  | good windows: 9/252
[1016] saved sub-1016_nr_timecourses.npz  | good windows: 90/252
[1002] saved sub-1002_nr_timecourses.npz  | good windows: 83/252
[1023] saved sub-1023_nr_timecourses.npz  | good windows: 58/252
[1034] saved sub-1034_nr_timecourses.npz  | good windows: 75/252
[1008] saved sub-1008_nr_timecourses.npz  | good windows: 75/252
[1041] saved sub-1041_nr_timecourses.npz  | good windows: 49/252
[3011] saved sub-3011_nr_timecourses.npz  | good windows: 61/252
[1001] saved sub-1001_nr_timecourses.npz  | good windows: 62/252
[3005] saved sub-3005_nr_timecourses.npz  | good windows: 43/252
[1042] saved sub-1042_nr_

In [16]:
# %% QC: plot rho / distance-corr / dynamic-range from nr_timecourses (no recompute)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def _smooth_1d(x, k=5):
    if k <= 1: return np.asarray(x, float)
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    ker = np.ones(k, float) / k
    return np.convolve(xp, ker, mode="valid")

def plot_qc_from_nr_timecourse(subject,
                               results_dir=RESULTS_DIR,
                               r_th=0.70, d_th=0.90, pr_frac=0.20,
                               smooth_k=5):
    f = results_dir / f"sub-{subject}_nr_timecourses.npz"
    if not f.exists():
        print(f"[{subject}] no nr_timecourses npz, skipping")
        return None

    with np.load(f, allow_pickle=False) as d:
        t  = d["time"].astype(float)
        rho = d["rho"].astype(float)
        distcorr = d["distcorr"].astype(float)
        prange = d["prange"].astype(float)

    pr_norm = prange / (np.nanmax(prange) + 1e-12)

    # smoothed copies (nicer onsets)
    rho_s  = _smooth_1d(rho, smooth_k)
    dist_s = _smooth_1d(distcorr, smooth_k)
    pr_s   = _smooth_1d(pr_norm, smooth_k)

    ok = (rho_s >= r_th) & (dist_s >= d_th) & (pr_s >= pr_frac)
    onset_time = float(t[np.argmax(ok)]) if np.any(ok) else np.nan

    in_range = (t >= TIME_RANGE[0]) & (t <= TIME_RANGE[1])
    if np.any(in_range):
        best_idx = int(np.argmax(rho[in_range]))
        best_time = float(t[in_range][best_idx])
    else:
        best_time = float(t[int(np.argmax(rho))])

    # plot
    plt.figure(figsize=(9,5))
    plt.plot(t, rho,      label='corr(x, contrast)')
    plt.plot(t, distcorr, label='distance corr')
    plt.plot(t, pr_norm,  label='pre-scale range (norm)')
    plt.axhline(0.7, ls='--', alpha=0.35); plt.axhline(0.9, ls='--', alpha=0.35)
    if np.isfinite(onset_time):
        plt.axvline(onset_time, color='k', lw=1, ls=':', alpha=0.8)
    plt.xlabel('Time (s)')
    plt.legend(); plt.tight_layout()
    out_png = results_dir / f"sub-{subject}_MDS_QC.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight"); plt.close()
    print(f"[{subject}] QC → {out_png}")

    return {
        "subject": subject,
        "onset_time": onset_time,
        "best_time": best_time,
        "rho_mean": float(np.nanmean(rho[in_range])) if np.any(in_range) else float(np.nanmean(rho)),
        "distcorr_mean": float(np.nanmean(distcorr[in_range])) if np.any(in_range) else float(np.nanmean(distcorr)),
        "prange_norm_mean": float(np.nanmean(pr_norm[in_range])) if np.any(in_range) else float(np.nanmean(pr_norm)),
    }

# run across cohort and save a tidy summary
qc_rows = []
for subj in SUBJECTS:
    row = plot_qc_from_nr_timecourse(subj, results_dir=RESULTS_DIR,
                                     r_th=0.70, d_th=0.90, pr_frac=0.20, smooth_k=5)
    if row is not None:
        qc_rows.append(row)

if qc_rows:
    df_qc = pd.DataFrame(qc_rows)
    out_csv = RESULTS_DIR / f"{ANALYSIS_NAME}_MDS_QC_summary.csv"
    df_qc.to_csv(out_csv, index=False)
    print("QC summary saved:", out_csv)


[1005] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1005_MDS_QC.png
[1007] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1007_MDS_QC.png
[1014] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1014_MDS_QC.png
[1004] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1004_MDS_QC.png
[1020] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1020_MDS_QC.png
[1017] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1017_MDS_QC.png
[1016] QC → /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/ecoc_results_01/sub-1016_MDS_QC.png
[1002]